# Lezione 8 — Pratica: analisi del modello

Corso "Deep Learning con Python e PyTorch da zero" — DIH DATAlife (Hub di
Innovazione Digitale Europeo)

In questa pratica lavori con due esercizi su un modello già addestrato
per riconoscere cifre scritte a mano del dataset MNIST (Modified National
Institute of Standards and Technology):

1. Completare il calcolo della precisione (accuracy) sull'insieme di test
   a partire dalle predizioni del modello
2. Salvare i pesi del modello con `torch.save`, "dimenticarlo" creando un
   modello nuovo non addestrato, e ricaricare i pesi salvati con
   `torch.load` e `load_state_dict`

Completa le celle contrassegnate con `# TODO`. Quando finisci ogni esercizio,
espandi la soluzione per verificare la tua risposta, ma cerca di risolverlo
da solo prima di guardarla.

## Preparazione comune

Esegui questa cella prima di iniziare uno dei due esercizi: carica
MNIST, definisce un modello piccolo e lo addestra per poche epoche per avere un
modello "già addestrato" con cui lavorare nei due esercizi. Non è
necessario che tu capisca ogni riga di questa cella: hai già visto questo
processo in lezioni precedenti, e in questa pratica ci interessa l'analisi,
non l'addestramento.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

transform = transforms.ToTensor()
train_dataset_completo = torchvision.datasets.MNIST(
    root="./datos", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(
    root="./datos", train=False, download=True, transform=transform)

N_TRAIN = 3000
train_dataset = Subset(train_dataset_completo, list(range(N_TRAIN)))

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


class RedDensa(nn.Module):
    def __init__(self):
        super().__init__()
        self.aplanar = nn.Flatten()
        self.capa1 = nn.Linear(28 * 28, 256)
        self.capa2 = nn.Linear(256, 128)
        self.capa_salida = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.aplanar(x)
        x = self.relu(self.capa1(x))
        x = self.relu(self.capa2(x))
        return self.capa_salida(x)


def entrenar(modelo, data_loader, num_epochs):
    funcion_perdida = nn.CrossEntropyLoss()
    optimizador = optim.Adam(modelo.parameters(), lr=1e-3)
    modelo.train()
    for _ in range(num_epochs):
        for imagenes, etiquetas in data_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            optimizador.zero_grad()
            salidas = modelo(imagenes)
            perdida = funcion_perdida(salidas, etiquetas)
            perdida.backward()
            optimizador.step()


net = RedDensa().to(device)
entrenar(net, train_loader, num_epochs=5)
net.eval()
print("Modelo entrenado y listo para analizar.")

## Esercizio 1 — Calcola la precisione sul set di test

Ya tienes el modelo `net` entrenado. Completa la celda siguiente para
calcular su precisión sobre `test_loader`, el conjunto de test que el
modelo no ha visto durante el entrenamiento.

**Pistas**

- `torch.argmax(salidas, dim=1)` te da, para cada imagen del lote, el índice
  de la clase con el valor más alto: la predicción del modelo
- Comparar `predicciones == etiquetas` te da un tensor de valores
  booleanos (`True`/`False`); `.sum().item()` cuenta cuántos son `True`
- La precisión es el número de aciertos dividido entre el número total de
  ejemplos evaluados
- Recuerda envolver el bucle en `torch.no_grad()`: solo estás evaluando, no
  entrenando

In [ ]:
aciertos = 0
total = 0

net.eval()
with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        salidas = net(imagenes)

        # TODO: calcula 'predicciones' a partir de 'salidas' usando
        # torch.argmax (la clase con el valor más alto, dim=1)
        predicciones = None  # TODO

        # TODO: suma a 'aciertos' el número de predicciones correctas de
        # este lote (compara 'predicciones' con 'etiquetas')
        # TODO: suma a 'total' el número de ejemplos de este lote
        # (etiquetas.size(0))

precision = None  # TODO: aciertos / total
print(f"Precisión sobre el conjunto de test: {precision:.2%}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
aciertos = 0
total = 0

net.eval()
with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        salidas = net(imagenes)

        # La clase predicha es el índice con el valor más alto en la salida
        predicciones = torch.argmax(salidas, dim=1)

        aciertos += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

precision = aciertos / total
print(f"Precisión sobre el conjunto de test: {precision:.2%}")

# Con un modelo entrenado unas pocas épocas sobre 3000 imágenes, la
# precisión suele rondar el 90%: nada mal para un modelo tan sencillo,
# aunque un modelo entrenado con las 60 000 imágenes completas llegaría
# más alto todavía.
```
</details>

## Esercizio 2 — Salva i pesi, "dimentica" il modello e ricargali

Vas a comprobar en primera persona por qué `torch.save` es tan útil: guarda
los pesos de `net` en un archivo, crea después un modelo **nuevo y sin
entrenar** (como si hubieras cerrado el notebook y perdido `net`), y
recupera en él los pesos guardados.

Completa los `# TODO`:

1. Guarda los pesos de `net` (su `state_dict`) en la ruta
   `ejercicio2_pesos.pt` con `torch.save`
2. Crea una instancia nueva de `RedDensa` llamada `modelo_nuevo` y muévela
   al mismo `device`
3. Antes de cargar los pesos, calcula la precisión de `modelo_nuevo` sobre
   `test_loader` (reutiliza la lógica del ejercicio 1) y guárdala en
   `precision_antes`
4. Carga los pesos guardados en `modelo_nuevo` con `load_state_dict` y
   `torch.load`
5. Vuelve a calcular la precisión y guárdala en `precision_despues`

**Pistas**

- `modelo_nuevo.load_state_dict(torch.load(ruta, map_location=device))` es
  la línea clave para recargar los pesos
- Antes de cargar los pesos, la precisión debería rondar el 10% (equivalente
  a elegir un dígito al azar entre diez posibles)
- Después de cargar los pesos, la precisión debería coincidir con la que
  obtuviste en el ejercicio 1

In [ ]:
def calcular_precision(modelo, data_loader):
    aciertos = 0
    total = 0
    modelo.eval()
    with torch.no_grad():
        for imagenes, etiquetas in data_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            predicciones = torch.argmax(salidas, dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
            total += etiquetas.size(0)
    return aciertos / total


ruta_pesos = "ejercicio2_pesos.pt"

# TODO 1: guarda los pesos de 'net' (net.state_dict()) en 'ruta_pesos'
# con torch.save


# TODO 2: crea una instancia nueva de RedDensa, sin entrenar, y muévela
# al mismo 'device' que 'net'
modelo_nuevo = None  # TODO

# TODO 3: calcula la precisión de 'modelo_nuevo' ANTES de cargar los pesos
# usando calcular_precision(modelo_nuevo, test_loader)
precision_antes = None  # TODO
print(f"Precisión antes de cargar los pesos: {precision_antes:.2%}")

# TODO 4: carga en 'modelo_nuevo' los pesos guardados en 'ruta_pesos' con
# modelo_nuevo.load_state_dict(torch.load(...))


# TODO 5: calcula la precisión de 'modelo_nuevo' DESPUÉS de cargar los pesos
precision_despues = None  # TODO
print(f"Precisión después de cargar los pesos: {precision_despues:.2%}")

<details>
<summary>💡 Vedi soluzione</summary>

```python
def calcular_precision(modelo, data_loader):
    aciertos = 0
    total = 0
    modelo.eval()
    with torch.no_grad():
        for imagenes, etiquetas in data_loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
            salidas = modelo(imagenes)
            predicciones = torch.argmax(salidas, dim=1)
            aciertos += (predicciones == etiquetas).sum().item()
            total += etiquetas.size(0)
    return aciertos / total


ruta_pesos = "ejercicio2_pesos.pt"

# Guardamos únicamente los pesos aprendidos de 'net'
torch.save(net.state_dict(), ruta_pesos)

# Modelo nuevo, con pesos inicializados al azar: es como si 'net' nunca
# hubiera existido
modelo_nuevo = RedDensa().to(device)

precision_antes = calcular_precision(modelo_nuevo, test_loader)
print(f"Precisión antes de cargar los pesos: {precision_antes:.2%}")

# Cargamos los pesos guardados dentro de la arquitectura nueva
modelo_nuevo.load_state_dict(torch.load(ruta_pesos, map_location=device))

precision_despues = calcular_precision(modelo_nuevo, test_loader)
print(f"Precisión después de cargar los pesos: {precision_despues:.2%}")

# precision_antes ronda el 10% (pesos aleatorios, equivalente a adivinar
# entre diez dígitos). precision_despues coincide con la precisión del
# ejercicio 1: hemos recuperado el conocimiento aprendido por 'net' sin
# volver a entrenar ni un solo paso.
```
</details>

## Per continuare a esercitarti

Se vuoi andare un po' oltre:

- Calcola anche la matrice di confusione su `test_loader` (come nel
  notebook di teoria) e scopri quale coppia di cifre confonde più il tuo
  modello
- Nell'esercizio 2, provare a caricare i pesi salvati in un'architettura
  leggermente diversa (ad esempio, cambiando la dimensione di `capa1` da 256 a
  300 neuroni) e osserva l'errore che PyTorch genera: è il modo più
  diretto per capire perché l'architettura deve coincidere esattamente
- Addestra `net` per qualche altra epoca sulle 3000 immagini e verifica se
  inizia a sovradattarsi: confronta la sua precisione su `train_loader`
  rispetto a `test_loader`

Ripassa il capitolo 08 del manuale se hai bisogno di rinfrescare la teoria
prima di provare.